# Data Q&A with Retrieval-Augmented Generation (RAG)

This notebook demonstrates how RAG improves LLM answers about your data.
We embed a data dictionary into ChromaDB, then the LLM retrieves relevant
context before answering — producing more accurate, grounded responses.

## Provider Configuration

Uncomment your preferred LLM and embedding providers below.

In [ ]:
from dotenv import load_dotenv
import seaborn as sns
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_anthropic import ChatAnthropic
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# LLM configuration — uncomment your preferred provider

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o")

# from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="llama3.2")

# Embedding configuration — uncomment your preferred provider

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# from langchain_ollama import OllamaEmbeddings
# embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Load Sample Data

The Palmer Penguins dataset — measurements of penguin species from three Antarctic islands.

In [ ]:
df = sns.load_dataset("penguins").dropna()
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

## Define the Data Dictionary

This is the domain knowledge we want the LLM to have access to.
In a real project, this could come from a wiki, a docs site, or a markdown file.

In [ ]:
data_dictionary = """
# Palmer Penguins Dataset — Data Dictionary

## Overview
This dataset contains measurements for three penguin species (Adelie, Chinstrap, Gentoo)
observed on three islands (Torgersen, Dream, Biscoe) in the Palmer Archipelago, Antarctica.
Data was collected by Dr. Kristen Gorman from 2007-2009.

## Columns

- **species**: Penguin species. One of: Adelie, Chinstrap, Gentoo.
  Adelie are the smallest and most widespread. Gentoo are the largest with
  distinctive orange bills. Chinstrap have a thin black line under the chin.
- **island**: Island name. Torgersen and Dream have Adelie penguins.
  Dream also has Chinstrap. Biscoe has Adelie and Gentoo.
- **bill_length_mm**: Length of the bill (culmen) in millimeters. Ranges ~32-60mm.
  Longer bills are typical of Gentoo and Chinstrap species.
- **bill_depth_mm**: Depth of the bill in millimeters. Ranges ~13-22mm.
  Adelie and Chinstrap have deeper bills relative to length than Gentoo.
- **flipper_length_mm**: Flipper length in millimeters. Ranges ~170-235mm.
  Gentoo have the longest flippers, correlating with their larger body size.
- **body_mass_g**: Body mass in grams. Ranges ~2700-6300g.
  Males are generally heavier than females across all species.
- **sex**: Penguin sex (Male or Female). Sexual dimorphism is most pronounced
  in body mass and flipper length.

## Common Analysis Patterns
- Species classification based on morphological measurements
- Sexual dimorphism analysis within and across species
- Island-species distribution patterns
- Bill shape (length vs depth ratio) as a species discriminator
- Body mass prediction from other measurements
"""

## Build the RAG Vector Store

Split the data dictionary into chunks, embed them, and store in ChromaDB.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.create_documents([data_dictionary])

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks into ChromaDB")

## Compare: Without RAG vs With RAG

First, let's ask the LLM directly (no context). Then ask with RAG retrieval.

In [ ]:
question = "Which penguin species has the deepest bill relative to its length, and why might that matter?"

response_no_rag = llm.invoke(question)
print("WITHOUT RAG:")
print(response_no_rag.content)

In [ ]:
system_prompt = (
    "You are a data analyst assistant. Use the following context from the data "
    "dictionary to answer questions about the penguins dataset. If the context "
    "doesn't contain the answer, say so.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("human", "{input}")]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response_rag = rag_chain.invoke({"input": question})
print("WITH RAG:")
print(response_rag["answer"])

## Ask More Questions

The RAG chain grounds answers in your data dictionary, producing more specific
and accurate responses than the LLM alone.

In [ ]:
for q in [
    "What islands can I find Chinstrap penguins on?",
    "If I want to predict body mass, which features should I use and why?",
    "How can I tell Adelie and Chinstrap apart using bill measurements?",
]:
    result = rag_chain.invoke({"input": q})
    print(f"Q: {q}")
    print(f"A: {result['answer']}\n")

## Notes

- ChromaDB stores vectors in-memory by default — for persistence, pass `persist_directory`
- Embedding cost is minimal: the data dictionary is small (a few hundred tokens)
- For larger data dictionaries, tune `chunk_size` and `k` for better retrieval
- Ollama embeddings (`nomic-embed-text`) work fully offline — no API key needed